# M2.4 Capacitance from Field Energy

Book pointer: chapter 8, sections 8.1 (capacitance), 8.2 (combining capacitors), 8.3 (energy stored in an electric field).

## 1. Recall

1. Define capacitance. Why is it a property of geometry alone for a linear dielectric?
2. Derive $C = \epsilon_0 A / d$ for parallel plates, and $C$ per unit length for coaxial cylinders of radii $a < b$.
3. Write the energy of a charged capacitor in three equivalent forms.
4. Write the field energy density and the total energy as a volume integral. Which form lets you compute $C$ without ever counting charge?
5. Series and parallel combination rules, and the one-line reason for each.

## 2. Principle to algorithm

Set one conductor to $V_0$ and the other to $0$, solve Laplace's equation, and integrate $\frac12\epsilon_0|\mathbf{E}|^2$ over the grid. That energy equals $\frac12 C V_0^2$, so $C = 2U/V_0^2$. The only ingredients are the solver from the previous notebook and a gradient. Because the 2D solver is uniform along $z$, every capacitance here is per unit depth.

The method is exact for any shape the grid can represent, which is the point: once the parallel-plate and coaxial cases agree with their closed forms, the same cells give the capacitance of geometries that have no closed form at all.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields, relax
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)

In [ ]:
def capacitance_from_energy(V, h, V0=1.0):
    return 2 * relax.field_energy(V, h) / V0**2

# Parallel plates: width W = 1, separation d = 1, sides fixed to the linear solution (no fringing).
n = 81
x = np.linspace(0, 1, n); h = x[1] - x[0]
V0 = np.zeros((n, n)); V0[:, 0] = V0[:, -1] = x; V0[-1, :] = 1.0
fixed = np.zeros((n, n), dtype=bool)
V = relax.solve_sparse(V0, fixed)
C = capacitance_from_energy(V, h)
print(f"parallel plates: C = {C:.6e} F/m    eps0 W/d = {EPS0:.6e} F/m    ratio {C / EPS0:.6f}")

### Coaxial cylinders

Inner radius $a$, outer radius $b$, closed form $C = 2\pi\epsilon_0 / \ln(b/a)$ per unit length. The circles are staircased onto the grid, so the error is set by how well the grid resolves the inner radius. Measure how the error falls as the grid is refined.

In [ ]:
def coax(n, a=0.25, b=1.0):
    xs = np.linspace(-b, b, n); hc = xs[1] - xs[0]
    X, Y = np.meshgrid(xs, xs, indexing="ij")
    R = np.hypot(X, Y)
    V0 = np.zeros((n, n)); fx = np.zeros((n, n), dtype=bool)
    V0[R <= a] = 1.0; fx[R <= a] = True        # inner conductor at 1 V
    fx[R >= b] = True                           # outer conductor at 0 V
    return relax.solve_sparse(V0, fx), hc, X, Y

C_exact = 2 * np.pi * EPS0 / np.log(1.0 / 0.25)
ns, errs = [41, 81, 161, 321], []
for m in ns:
    Vc, hc, _, _ = coax(m)
    errs.append(abs(capacitance_from_energy(Vc, hc) / C_exact - 1))
    print(f"n = {m:4d}  h = {hc:.4f}  C / C_exact = {1 + errs[-1]:.5f}")
plt.loglog(ns, errs, "o-"); plt.xlabel("grid points per side"); plt.ylabel("relative error in C"); plt.grid(True, which="both", alpha=0.3); plt.show()

### Exercise: fringing fields

Replace the body of `finite_plates` so that it builds two finite plates of width $W$ and separation $d$ inside a large grounded box, with free space around them. With fringing the capacitance exceeds $\epsilon_0 W/d$. The check requires the ratio to be above one and below two for $W/d = 4$. Note the grounded box also stores field energy, so the number you get is the capacitance of the plates in that box, not in free space. Enlarging the box shows how quickly that matters.

In [ ]:
def finite_plates(n=241, W=1.0, d=0.25, box=3.0):
    """Plates at +0.5 V and -0.5 V (so V0 = 1 between them) inside a grounded box of half-width box."""
    xs = np.linspace(-box, box, n); hp = xs[1] - xs[0]
    X, Y = np.meshgrid(xs, xs, indexing="ij")
    V0 = np.zeros((n, n)); fx = np.zeros((n, n), dtype=bool)
    plate = np.abs(X) <= W / 2
    top = plate & (np.abs(Y - d / 2) < hp / 2); bot = plate & (np.abs(Y + d / 2) < hp / 2)
    V0[top] = 0.5; V0[bot] = -0.5; fx[top | bot] = True       # replace this construction with your own
    return relax.solve_sparse(V0, fx), hp


Vf, hp = finite_plates()
C_f = capacitance_from_energy(Vf, hp, V0=1.0)
ratio = C_f / (EPS0 * 1.0 / 0.25)
print(f"finite plates W/d = 4:  C / (eps0 W/d) = {ratio:.3f}")
assert 1.0 < ratio < 2.0

xs = np.linspace(-3, 3, 241)
Xf, Yf = np.meshgrid(xs, xs, indexing="ij")
plt.contour(Xf, Yf, Vf, levels=np.linspace(-0.45, 0.45, 19), cmap="coolwarm")
plt.xlim(-1.5, 1.5); plt.ylim(-1, 1); plt.gca().set_aspect("equal"); plt.title("fringing field of finite plates"); plt.show()

## 4. Verify

Two capacitors in series. Three plates at $V = 1$, free, and $0$ with gaps $d_1$ and $d_2$: the middle plate floats. In a solver with Dirichlet conductors a floating conductor is awkward, so use the field route instead. Compute the energy with the middle plate at a trial potential $V_m$, and recall that the true floating potential minimises the energy. Scan $V_m$, find the minimum, and compare the result to $1/C = 1/C_1 + 1/C_2$.

In [ ]:
n = 121
x = np.linspace(0, 1, n); h = x[1] - x[0]
i_mid = 40                                          # middle plate at x = 1/3 -> d1 = 1/3, d2 = 2/3
C1, C2 = EPS0 / (1 / 3), EPS0 / (2 / 3)
C_series = 1 / (1 / C1 + 1 / C2)

def energy_with_middle_at(Vm):
    V0 = np.zeros((n, n)); fx = np.zeros((n, n), dtype=bool)
    V0[-1, :] = 1.0; V0[i_mid, :] = Vm; fx[i_mid, :] = True
    # sides: piecewise-linear between the three plates, so there is no fringing
    V0[:, 0] = V0[:, -1] = np.where(x <= x[i_mid], Vm * x / x[i_mid], Vm + (1 - Vm) * (x - x[i_mid]) / (1 - x[i_mid]))
    return relax.field_energy(relax.solve_sparse(V0, fx), h)

Vms = np.linspace(0.3, 1.0, 36)
Us = np.array([energy_with_middle_at(v) for v in Vms])
k = np.argmin(Us)
a_, b_, c_ = np.polyfit(Vms[k - 2:k + 3], Us[k - 2:k + 3], 2); Vm_star = -b_ / (2 * a_)
U_star = energy_with_middle_at(Vm_star)
C_num = 2 * U_star / 1.0**2
plt.plot(Vms, Us, "o-"); plt.axvline(Vm_star, color="C3", ls="--", label=f"minimum at V_m = {Vm_star:.3f}")
plt.xlabel("middle plate potential"); plt.ylabel("field energy per unit depth"); plt.legend(); plt.show()
print(f"floating potential {Vm_star:.4f}  (voltage divider gives {C1 / (C1 + C2):.4f})")
print(f"C from energy {C_num:.5e}   series formula {C_series:.5e}   ratio {C_num / C_series:.5f}")
assert abs(C_num / C_series - 1) < 1e-3

## 5. Explore and rebuild

Explore. Take the coaxial geometry and move the inner conductor off centre by a distance $e$. Plot $C(e)/C(0)$ as $e$ goes from $0$ to $0.9(b-a)$. The closed form for eccentric cylinders is $C = 2\pi\epsilon_0 / \cosh^{-1}\!\big((a^2 + b^2 - e^2)/(2ab)\big)$. Compare, and explain physically why the capacitance rises.

Rebuild. Close this notebook and write `field_energy` and `capacitance_from_energy` from scratch, then reproduce the parallel-plate and coaxial checks.